# GRPO Rollout Path with vLLM

This notebook demonstrates the external `vLLM` rollout service pattern that can be paired with Trainer-based GRPO on Red Hat OpenShift AI.

The default flow validates the customer-facing rollout path:
- deploy a namespace-scoped `vLLM` service
- confirm health and OpenAI-compatible inference
- validate the rollout control-plane endpoints needed for later weight-sync work

An optional advanced section is included for native weight-sync validation, but it is intentionally disabled by default.


## Install dependencies


In [ ]:
# !python3 -m pip install -U kubeflow kubernetes openai requests transformers vllm


In [ ]:
import json
import os
import time

import requests
from kubernetes import client as k8s
from openai import OpenAI

## Configure cluster access and rollout service settings


In [ ]:
api_server = os.getenv("OPENSHIFT_API_URL")
token = os.getenv("NOTEBOOK_USER_TOKEN")
NAMESPACE = os.getenv("NOTEBOOK_NAMESPACE", "<your-namespace>")
VLLM_NAME = "grpo-vllm-rollout"
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
IMAGE = os.getenv("VLLM_IMAGE", "vllm/vllm-openai:latest")
API_KEY = os.getenv("VLLM_API_KEY", "dummy-token")
RUN_OPTIONAL_VALIDATION = (
    os.getenv("RUN_OPTIONAL_VALIDATION", "false").lower() == "true"
)
VLLM_LOAD_FORMAT = os.getenv("VLLM_LOAD_FORMAT", "auto")
WEIGHT_TRANSFER_BACKEND = os.getenv("VLLM_WEIGHT_TRANSFER_BACKEND", "nccl")
WEIGHT_SYNC_PORT = int(os.getenv("VLLM_WEIGHT_SYNC_PORT", "29501"))
ENABLE_FULL_WEIGHT_SYNC = (
    os.getenv("ENABLE_FULL_WEIGHT_SYNC", "false").lower() == "true"
)

if not api_server or not token:
    raise RuntimeError("OPENSHIFT_API_URL and NOTEBOOK_USER_TOKEN must be set.")
if NAMESPACE == "<your-namespace>":
    raise RuntimeError(
        "Set NOTEBOOK_NAMESPACE to your OpenShift AI project namespace before running this notebook."
    )

configuration = k8s.Configuration()
configuration.host = api_server
configuration.verify_ssl = True
for candidate in [
    "/var/run/secrets/kubernetes.io/serviceaccount/ca.crt",
    os.getenv("SSL_CERT_FILE"),
    os.getenv("REQUESTS_CA_BUNDLE"),
]:
    if candidate and os.path.exists(candidate):
        configuration.ssl_ca_cert = candidate
        break
configuration.api_key = {"authorization": f"Bearer {token}"}
api_client = k8s.ApiClient(configuration)
apps = k8s.AppsV1Api(api_client)
core = k8s.CoreV1Api(api_client)
networking = k8s.NetworkingV1Api(api_client)
workbench_pod_name = os.getenv("WORKBENCH_POD_NAME") or os.getenv("HOSTNAME")
workbench_name = (
    os.getenv("WORKBENCH_NAME")
    or os.getenv("NOTEBOOK_NAME")
    or (
        "-".join(workbench_pod_name.split("-")[:-1])
        if workbench_pod_name and "-" in workbench_pod_name
        else workbench_pod_name
    )
)
service_host = f"{VLLM_NAME}.{NAMESPACE}.svc.cluster.local"
service_root = f"http://{service_host}:8000"
base_url = f"{service_root}/v1"
health_url = f"{service_root}/health"
if not workbench_pod_name:
    raise RuntimeError(
        "WORKBENCH_POD_NAME or HOSTNAME must be set to identify the trainer-side workbench pod."
    )
workbench_pod = core.read_namespaced_pod(name=workbench_pod_name, namespace=NAMESPACE)
workbench_pod_ip = workbench_pod.status.pod_ip
print(f"Using rollout service host: {service_host}")
print(f"Optional validation enabled: {RUN_OPTIONAL_VALIDATION}")
print(f"Weight transfer backend: {WEIGHT_TRANSFER_BACKEND}")
print(f"Weight sync port: {WEIGHT_SYNC_PORT}")
print(f"Workbench pod: {workbench_pod_name} ({workbench_pod_ip})")
print(f"Workbench selector label: notebook-name={workbench_name}")
print(f"Full weight sync enabled: {ENABLE_FULL_WEIGHT_SYNC}")

## Create the vLLM rollout Deployment and Service


In [ ]:
cache_dir = "/tmp/hf-cache"
home_dir = "/tmp/vllm-home"
flashinfer_dir = "/tmp/flashinfer-cache"

deployment = {
    "apiVersion": "apps/v1",
    "kind": "Deployment",
    "metadata": {"name": VLLM_NAME, "namespace": NAMESPACE},
    "spec": {
        "replicas": 1,
        "selector": {"matchLabels": {"app": VLLM_NAME}},
        "template": {
            "metadata": {"labels": {"app": VLLM_NAME}},
            "spec": {
                "containers": [
                    {
                        "name": "vllm",
                        "image": IMAGE,
                        "args": [
                            MODEL_ID,
                            "--host",
                            "0.0.0.0",
                            "--port",
                            "8000",
                            "--gpu-memory-utilization",
                            "0.8",
                            "--max-model-len",
                            "4096",
                            "--enforce-eager",
                            "--load-format",
                            VLLM_LOAD_FORMAT,
                            "--weight-transfer-config",
                            json.dumps({"backend": WEIGHT_TRANSFER_BACKEND}),
                        ],
                        "ports": [{"containerPort": 8000, "name": "http"}],
                        "env": [
                            {"name": "HF_TOKEN", "value": os.getenv("HF_TOKEN", "")},
                            {"name": "HOME", "value": home_dir},
                            {"name": "HF_HOME", "value": cache_dir},
                            {"name": "HUGGINGFACE_HUB_CACHE", "value": cache_dir},
                            {"name": "TRANSFORMERS_CACHE", "value": cache_dir},
                            {"name": "XDG_CACHE_HOME", "value": "/tmp"},
                            {"name": "VLLM_CACHE_ROOT", "value": "/tmp/vllm-cache"},
                            {"name": "VLLM_CONFIG_ROOT", "value": "/tmp/vllm-config"},
                            {
                                "name": "FLASHINFER_WORKSPACE_DIR",
                                "value": flashinfer_dir,
                            },
                            {"name": "VLLM_SERVER_DEV_MODE", "value": "1"},
                        ],
                        "resources": {
                            "requests": {
                                "cpu": "4",
                                "memory": "24Gi",
                                "nvidia.com/gpu": "1",
                            },
                            "limits": {
                                "cpu": "4",
                                "memory": "24Gi",
                                "nvidia.com/gpu": "1",
                            },
                        },
                        "volumeMounts": [
                            {"name": "hf-cache", "mountPath": cache_dir},
                            {"name": "vllm-home", "mountPath": home_dir},
                            {"name": "flashinfer-cache", "mountPath": flashinfer_dir},
                        ],
                    }
                ],
                "volumes": [
                    {"name": "hf-cache", "emptyDir": {}},
                    {"name": "vllm-home", "emptyDir": {}},
                    {"name": "flashinfer-cache", "emptyDir": {}},
                ],
            },
        },
    },
}
service = {
    "apiVersion": "v1",
    "kind": "Service",
    "metadata": {"name": VLLM_NAME, "namespace": NAMESPACE},
    "spec": {
        "selector": {"app": VLLM_NAME},
        "ports": [{"port": 8000, "targetPort": 8000, "name": "http"}],
    },
}

try:
    apps.create_namespaced_deployment(namespace=NAMESPACE, body=deployment)
except Exception:
    apps.patch_namespaced_deployment(
        name=VLLM_NAME, namespace=NAMESPACE, body=deployment
    )

try:
    core.create_namespaced_service(namespace=NAMESPACE, body=service)
except Exception:
    core.patch_namespaced_service(name=VLLM_NAME, namespace=NAMESPACE, body=service)

print(f"Created or updated Deployment/Service for {VLLM_NAME}")

## Allow rollout-to-workbench traffic for native weight sync

Native weight sync requires the `vLLM` rollout pod to connect back to the GPU workbench on a fixed trainer rendezvous port. This cell creates a narrow NetworkPolicy to allow only that traffic.


In [ ]:
network_policy_name = f"{workbench_name}-vllm-weight-sync"
network_policy = {
    "apiVersion": "networking.k8s.io/v1",
    "kind": "NetworkPolicy",
    "metadata": {"name": network_policy_name, "namespace": NAMESPACE},
    "spec": {
        "podSelector": {"matchLabels": {"notebook-name": workbench_name}},
        "policyTypes": ["Ingress"],
        "ingress": [
            {
                "from": [{"podSelector": {"matchLabels": {"app": VLLM_NAME}}}],
                "ports": [{"protocol": "TCP", "port": WEIGHT_SYNC_PORT}],
            }
        ],
    },
}
try:
    networking.create_namespaced_network_policy(
        namespace=NAMESPACE, body=network_policy
    )
except Exception:
    networking.patch_namespaced_network_policy(
        name=network_policy_name, namespace=NAMESPACE, body=network_policy
    )
print(
    f"Created or updated NetworkPolicy {network_policy_name} for TCP/{WEIGHT_SYNC_PORT}"
)

## Wait for vLLM and validate the API


In [ ]:
for _ in range(60):
    try:
        response = requests.get(health_url, timeout=5)
        if response.status_code == 200:
            print("vLLM health check passed")
            break
    except Exception:
        pass
    time.sleep(10)
else:
    raise RuntimeError("vLLM service did not become healthy in time")

## Optional native weight-sync validation

This section is disabled by default. Enable it only from a GPU-enabled workbench when you explicitly want to validate native weight transfer after the rollout path is already working. It is not required for the standard customer-facing rollout example.


In [ ]:
if not RUN_OPTIONAL_VALIDATION:
    print(
        "Skipping optional rollout API and control-plane validation. Set RUN_OPTIONAL_VALIDATION=true to execute these checks."
    )
else:
    client = OpenAI(base_url=base_url, api_key=API_KEY)
    response = client.chat.completions.create(
        model=MODEL_ID,
        messages=[
            {
                "role": "user",
                "content": "Solve: If Sarah has 3 apples and gets 2 more, how many apples does she have? End with #### <number>.",
            }
        ],
        max_tokens=64,
    )
    print(response.choices[0].message.content)
    world_size_response = requests.get(f"{service_root}/get_world_size", timeout=10)
    world_size_response.raise_for_status()
    print("vLLM world size:", world_size_response.json())
    pause_response = requests.post(f"{service_root}/pause", timeout=30)
    pause_response.raise_for_status()
    print("pause status:", pause_response.status_code)
    resume_response = requests.post(f"{service_root}/resume", timeout=30)
    resume_response.raise_for_status()
    print("resume status:", resume_response.status_code)

## Optional full native weight-sync validation

Set `ENABLE_FULL_WEIGHT_SYNC=true` only when this notebook runs from a GPU-enabled workbench. This section uses a fixed rendezvous port plus the NetworkPolicy created above so the `vLLM` rollout pod can connect back to the trainer-side process.


In [ ]:
if not ENABLE_FULL_WEIGHT_SYNC:
    print(
        "Skipping full native weight sync validation. Set ENABLE_FULL_WEIGHT_SYNC=true on a GPU-enabled workbench to run this step."
    )
else:
    import threading

    import torch
    from transformers import AutoModelForCausalLM
    from vllm.distributed.weight_transfer.nccl_engine import (
        NCCLTrainerSendWeightsArgs,
        NCCLWeightTransferEngine,
    )

    if not torch.cuda.is_available():
        raise RuntimeError("Full weight sync requires a GPU-enabled workbench.")

    inference_world_size = requests.get(
        f"{service_root}/get_world_size", timeout=10
    ).json()["world_size"]
    total_world_size = inference_world_size + 1
    torch.cuda.set_device(0)
    trainer_device = "cuda:0"
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    print("loading trainer model", MODEL_ID, dtype, trainer_device)
    train_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=dtype)
    train_model.to(trainer_device)

    master_address = workbench_pod_ip
    master_port = WEIGHT_SYNC_PORT
    rank_offset = 1
    print("init", master_address, master_port, rank_offset, total_world_size)

    def init_engine():
        payload = {
            "init_info": {
                "master_address": master_address,
                "master_port": master_port,
                "rank_offset": rank_offset,
                "world_size": total_world_size,
            }
        }
        r = requests.post(
            f"{service_root}/init_weight_transfer_engine", json=payload, timeout=120
        )
        r.raise_for_status()
        print("init_weight_transfer_engine", r.status_code)

    init_thread = threading.Thread(target=init_engine)
    init_thread.start()
    model_update_group = NCCLWeightTransferEngine.trainer_init({
        "master_address": master_address,
        "master_port": master_port,
        "world_size": total_world_size,
    })
    init_thread.join()

    requests.post(f"{service_root}/pause", timeout=30).raise_for_status()
    requests.post(
        f"{service_root}/start_weight_update",
        json={"is_checkpoint_format": True},
        timeout=60,
    ).raise_for_status()

    names, dtype_names, shapes = [], [], []
    for name, param in train_model.named_parameters():
        names.append(name)
        dtype_names.append(str(param.dtype).split(".")[-1])
        shapes.append(list(param.shape))

    def update_metadata():
        payload = {
            "update_info": {
                "names": names,
                "dtype_names": dtype_names,
                "shapes": shapes,
                "packed": True,
            }
        }
        r = requests.post(f"{service_root}/update_weights", json=payload, timeout=300)
        r.raise_for_status()
        print("update_weights", r.status_code)

    update_thread = threading.Thread(target=update_metadata)
    update_thread.start()
    NCCLWeightTransferEngine.trainer_send_weights(
        iterator=train_model.named_parameters(),
        trainer_args=NCCLTrainerSendWeightsArgs(group=model_update_group, packed=True),
    )
    update_thread.join()

    requests.post(
        f"{service_root}/finish_weight_update", json={}, timeout=60
    ).raise_for_status()
    requests.post(f"{service_root}/resume", timeout=30).raise_for_status()
    print(
        "after",
        client.chat.completions
        .create(
            model=MODEL_ID,
            messages=[{"role": "user", "content": "The capital of France is"}],
            max_tokens=32,
        )
        .choices[0]
        .message.content,
    )
    print("weight_sync_done")